# 02 · The Spark pipeline

**Question:** how do we turn ~470 text files (or ~130,000 for the full archive) into something fast to query, and how do we know the result is sound?

`python -m ghcnd_quality.build` reads the raw `.dly` files with the parser from notebook 01 and writes Parquet partitioned by element and decade. This notebook looks at the result, measures what the conversion buys, and runs basic integrity checks.

In [1]:
import io, os, time, pathlib, contextlib
import pandas as pd
from ghcnd_quality.spark import get_spark
from ghcnd_quality.dly import read_dly
from ghcnd_quality.audit import register_views, run_sql

pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30)
RAW, PARQ = pathlib.Path("../data/raw"), pathlib.Path("../data/parquet")
spark = get_spark("nb02"); register_views(spark, PARQ)
print("Spark", spark.version, "| cores:", spark.sparkContext.defaultParallelism)

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=40689 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/etc/ssl/certs/java/cacerts -Djavax.net.ssl.trustStoreType=JKS -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=40689 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/10 18:35:10 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/10/10 18:35:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/10/10 18:35:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0 | cores: 2


## Parse once, query many

The same aggregate (observations per element), once from the raw text files and once from Parquet. Single run each on this machine, so read it as an order of magnitude, not a benchmark:

In [2]:
t = time.time()
raw_counts = read_dly(spark, str(RAW / "ghcnd_all" / "*.dly")).groupBy("element").count().toPandas()
raw_s = time.time() - t

t = time.time()
pq_counts = spark.sql("select element, count(*) as count from daily group by element").toPandas()
pq_s = time.time() - t

print(f"raw .dly text : {raw_s:6.1f} s   (parse 31 slots per line, every time)")
print(f"parquet       : {pq_s:6.1f} s   ({raw_s / pq_s:.0f}x faster)")
cmp = raw_counts.merge(pq_counts, on="element", suffixes=("_raw", "_parquet")).sort_values("element")
assert (cmp["count_raw"] == cmp["count_parquet"]).all()
cmp

26/10/10 18:35:18 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/raw/ghcnd_all/*.dly.
java.io.FileNotFoundException: File ../data/raw/ghcnd_all/*.dly does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.

/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/10/10 18:35:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


raw .dly text :   97.6 s   (parse 31 slots per line, every time)
parquet       :    1.6 s   (61x faster)


,element,count_raw,count_parquet
2,PRCP,9866516,9866516
0,SNOW,651959,651959
1,SNWD,1706115,1706115
3,TAVG,7452516,7452516
5,TMAX,9897013,9897013
4,TMIN,9784101,9784101


Both paths return identical counts per element, so the conversion lost nothing.

## Layout on disk

Partitioned by `element` and `decade`, with a `repartition` before the write so each partition is written by one task. Without it, every task writes a file into every partition and the small-files problem returns at the other end.

In [3]:
rows = []
for el in sorted(p for p in os.listdir(PARQ / "daily") if p.startswith("element=")):
    files = [os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(PARQ / "daily" / el)
             for f in fs if f.endswith(".parquet")]
    rows.append({"partition": el, "decades": len(os.listdir(PARQ / "daily" / el)),
                 "parquet_files": len(files), "size_mb": round(sum(files) / 1e6, 1)})
layout = pd.DataFrame(rows)
print(f"{len(list((RAW / 'ghcnd_all').glob('*.dly')))} raw files -> {layout.parquet_files.sum()} parquet files, "
      f"{layout.size_mb.sum():.0f} MB")
layout

474 raw files -> 102 parquet files, 145 MB


,partition,decades,parquet_files,size_mb
0,element=PRCP,20,20,31.5
1,element=SNOW,14,14,1.1
2,element=SNWD,15,15,3.8
3,element=TAVG,15,15,29.3
4,element=TMAX,19,19,39.5
5,element=TMIN,19,19,39.9


A filter on the partition columns reads only the matching directories. The plan shows it:

In [4]:
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    spark.sql("select count(*) from daily where element = 'TMAX' and decade = 1990").explain(mode="formatted")
for line in buf.getvalue().splitlines():
    if "PartitionFilters" in line or "Location" in line:
        print(line.strip()[:200])

Location: InMemoryFileIndex [file:/home/claude/ghcnd-quality-spark/data/parquet/daily]
PartitionFilters: [isnotnull(element#10), isnotnull(decade#11), (element#10 = TMAX), (decade#11 = 1990)]


## Integrity checks on the result

A parsed table is only useful if its basic invariants hold. Primary key first: one value per station, day and element.

In [5]:
spark.sql('''
    select count(*) as duplicate_keys from (
        select station_id, date, element from daily group by station_id, date, element having count(*) > 1)
''').toPandas()

/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,duplicate_keys
0,0


In [6]:
spark.sql('''
    select count(*)                               as observations,
           count_if(station_id is null)           as null_station,
           count_if(date is null)                 as null_date,
           count_if(value is null)                as null_value,
           min(date)                              as first_date,
           max(date)                              as last_date
    from daily
''').toPandas()

/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,observations,null_station,null_date,null_value,first_date,last_date
0,39358220,0,0,0,1838-01-01,2026-10-08


Value ranges per element. These are the raw extremes **including** observations NOAA flagged, which is the point of keeping them:

In [7]:
spark.sql('''
    select element,
           min(value_native) as min_value,
           max(value_native) as max_value,
           count(*)          as observations
    from daily group by element order by element
''').toPandas()

/home/claude/venv-spark/lib/python3.13/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,element,min_value,max_value,observations
0,PRCP,-3.0,2189.0,9866516
1,SNOW,0.0,1778.0,651959
2,SNWD,-30.0,11900.0,1706115
3,TAVG,-47.9,43.3,7452516
4,TMAX,-99.0,999.0,9897013
5,TMIN,-999.0,999.0,9784101


Next: [03 · Quality audit](03_quality_audit.ipynb)